# Notebook 02 — FGSM & PGD Baselines

Runs two standard gradient-based adversarial attacks on all three trained models
and saves the results for comparison in Notebook 05.

| Attack | Type | What it gives us |
|---|---|---|
| **FGSM** | Single-step gradient | Fast upper bound on ε — weakest baseline |
| **PGD** | Multi-step gradient | Tight upper bound on ε — strongest gradient baseline |

**Both provide only upper bounds** — they find adversarial examples but cannot
certify that none exist within a given radius. This is the key limitation
our concolic framework addresses by also providing a lower bound.

**Requires:** `models/small_mnist.pt`, `models/medium_mnist.pt`, `models/large_cifar.pt`  
*(Run Notebook 01 first)*

**Outputs saved to:**
```
results/baselines_small_mnist.json
results/baselines_medium_mnist.json
results/baselines_large_cifar.json
results/baselines_summary.json
```

## 0 — Install dependencies

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'scipy', 'tqdm', 'numpy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)

import torch
print(f'PyTorch : {torch.__version__}')

PyTorch : 2.12.0+cpu


## 1 — Imports & configuration

In [2]:
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from pathlib import Path
from tqdm import tqdm

# ── path setup ────────────────────────────────────────────────────────────────
REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.network_definitions import load_model
from utils.metrics import (
    compute_accuracy, compute_robustness_stats,
    get_correctly_classified_samples,
    save_results, print_table,
)

MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE      = 'cpu'
NUM_WORKERS = 0
SEED        = 42
N_SAMPLES   = 100    # number of test inputs per model — enough for statistical validity

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Evaluating {N_SAMPLES} correctly-classified samples per model')

Evaluating 100 correctly-classified samples per model


## 2 — Attack implementations

In [3]:
# ── FGSM ──────────────────────────────────────────────────────────────────────
def fgsm_attack(model, x, label, epsilon):
    """
    Fast Gradient Sign Method (Goodfellow et al., 2015).
    Single-step gradient attack under L∞ norm.

    x_adv = x + epsilon * sign(∇_x L(f(x), y))

    Returns: adversarial example x_adv as numpy array.
    """
    x_t = torch.tensor(x, dtype=torch.float32, requires_grad=True)
    logits = model(x_t.unsqueeze(0))
    loss   = nn.CrossEntropyLoss()(logits, torch.tensor([label]))
    loss.backward()
    x_adv = x + epsilon * x_t.grad.sign().numpy()
    return x_adv


def fgsm_perturbation_radius(model, x, label,
                              eps_min=0.001, eps_max=1.0, n_steps=30):
    """
    Binary search for the smallest FGSM epsilon that flips the prediction.
    This gives a tighter upper bound than a fixed epsilon.

    Returns: (eps_found, x_adv) or (eps_max, None) if no flip found.
    """
    model.eval()
    lo, hi = eps_min, eps_max
    best_eps = eps_max
    best_adv = None

    for _ in range(n_steps):
        mid   = (lo + hi) / 2
        x_adv = fgsm_attack(model, x, label, mid)
        with torch.no_grad():
            pred = model(torch.tensor(x_adv, dtype=torch.float32).unsqueeze(0))
            pred = pred.argmax().item()
        if pred != label:
            best_eps = mid
            best_adv = x_adv
            hi = mid
        else:
            lo = mid

    return best_eps, best_adv


# ── PGD ───────────────────────────────────────────────────────────────────────
def pgd_attack(model, x, label, epsilon, alpha=None, n_iter=40):
    """
    Projected Gradient Descent attack (Madry et al., 2018).
    Multi-step gradient attack under L∞ norm.

    x_0   = x + uniform noise in [-epsilon, epsilon]
    x_t+1 = clip(x_t + alpha * sign(∇_x L), x-eps, x+eps)

    Args:
        epsilon : L∞ perturbation budget
        alpha   : step size (default: epsilon * 2.5 / n_iter)
        n_iter  : number of gradient steps

    Returns: adversarial example x_adv as numpy array.
    """
    if alpha is None:
        alpha = epsilon * 2.5 / n_iter

    x_t   = torch.tensor(x, dtype=torch.float32)
    x_orig = x_t.clone()

    # random start within epsilon ball
    x_t = x_t + torch.zeros_like(x_t).uniform_(-epsilon, epsilon)

    for _ in range(n_iter):
        x_t = x_t.detach().requires_grad_(True)
        logits = model(x_t.unsqueeze(0))
        loss   = nn.CrossEntropyLoss()(logits, torch.tensor([label]))
        loss.backward()
        with torch.no_grad():
            x_t = x_t + alpha * x_t.grad.sign()
            # project back into L∞ ball around original x
            x_t = torch.clamp(x_t, x_orig - epsilon, x_orig + epsilon)

    return x_t.detach().numpy()


def pgd_perturbation_radius(model, x, label,
                             eps_min=0.001, eps_max=1.0,
                             n_search=20, n_iter=40):
    """
    Binary search for the smallest PGD epsilon that flips the prediction.
    More reliable than FGSM because PGD is a stronger attack.

    Returns: (eps_found, x_adv) or (eps_max, None) if no flip found.
    """
    model.eval()
    lo, hi   = eps_min, eps_max
    best_eps = eps_max
    best_adv = None

    for _ in range(n_search):
        mid   = (lo + hi) / 2
        x_adv = pgd_attack(model, x, label, epsilon=mid, n_iter=n_iter)
        with torch.no_grad():
            pred = model(torch.tensor(x_adv, dtype=torch.float32).unsqueeze(0))
            pred = pred.argmax().item()
        if pred != label:
            best_eps = mid
            best_adv = x_adv
            hi = mid
        else:
            lo = mid

    return best_eps, best_adv


print('Attack functions defined.')

Attack functions defined.


## 3 — Batch evaluation function

In [4]:
def evaluate_baselines(model, X, y, model_name=''):
    """
    Run FGSM and PGD perturbation radius estimation on all samples.

    Returns dict with per-sample results and aggregate stats.
    """
    model.eval()
    fgsm_results = []
    pgd_results  = []

    bar = tqdm(zip(X, y), total=len(X), desc=f'{model_name}', unit='sample')

    for x, label in bar:
        label = int(label)
        t0    = time.time()

        # ── FGSM ─────────────────────────────────────────────────────────────
        fgsm_eps, fgsm_adv = fgsm_perturbation_radius(
            model, x, label, eps_min=0.001, eps_max=1.0, n_steps=25
        )
        fgsm_time = time.time() - t0

        # ── PGD ──────────────────────────────────────────────────────────────
        t1 = time.time()
        pgd_eps, pgd_adv = pgd_perturbation_radius(
            model, x, label,
            eps_min=0.001, eps_max=1.0,
            n_search=15, n_iter=30
        )
        pgd_time = time.time() - t1

        fgsm_results.append({
            'eps_upper'   : float(fgsm_eps),
            'eps_lower'   : 0.0,          # FGSM gives no lower bound
            'adv_found'   : fgsm_adv is not None,
            'runtime_sec' : fgsm_time,
        })
        pgd_results.append({
            'eps_upper'   : float(pgd_eps),
            'eps_lower'   : 0.0,          # PGD gives no lower bound
            'adv_found'   : pgd_adv is not None,
            'runtime_sec' : pgd_time,
        })

        bar.set_postfix(
            fgsm=f'{fgsm_eps:.3f}',
            pgd=f'{pgd_eps:.3f}',
        )

    # ── aggregate stats ───────────────────────────────────────────────────────
    def agg(results):
        eps    = np.array([r['eps_upper']   for r in results])
        times  = np.array([r['runtime_sec'] for r in results])
        found  = np.array([r['adv_found']   for r in results])
        return {
            'mean_eps_upper'            : float(np.mean(eps)),
            'std_eps_upper'             : float(np.std(eps)),
            'median_eps_upper'          : float(np.median(eps)),
            'mean_eps_lower'            : 0.0,
            'mean_runtime_sec'          : float(np.mean(times)),
            'total_runtime_sec'         : float(np.sum(times)),
            'fraction_adversarial_found': float(np.mean(found)),
            'n_samples'                 : len(results),
        }

    return {
        'fgsm': {'per_sample': fgsm_results, 'stats': agg(fgsm_results)},
        'pgd' : {'per_sample': pgd_results,  'stats': agg(pgd_results)},
    }


print('Evaluation function defined.')

Evaluation function defined.


## 4 — Load datasets (test splits only)

In [5]:
mnist_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_test = datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf)

cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
cifar_test = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_tf)

# flatten to numpy arrays for attack functions
def dataset_to_numpy(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    X, y   = next(iter(loader))
    return X.view(X.size(0), -1).numpy(), y.numpy()

print('Loading datasets...')
X_mnist, y_mnist = dataset_to_numpy(mnist_test)
X_cifar, y_cifar = dataset_to_numpy(cifar_test)
print(f'MNIST  test : {X_mnist.shape}')
print(f'CIFAR  test : {X_cifar.shape}')

Loading datasets...
MNIST  test : (10000, 784)
CIFAR  test : (10000, 3072)


## 5 — Run on SmallMLP (MNIST)
**Expected time: ~10 min**

In [6]:
print('Loading small_mnist model...')
small_model = load_model(str(MODELS_DIR / 'small_mnist.pt'))

# select N_SAMPLES correctly-classified inputs
X_small, y_small = get_correctly_classified_samples(
    small_model, X_mnist, y_mnist, N_SAMPLES, seed=SEED
)
print(f'Selected {len(X_small)} correctly-classified samples\n')

results_small = evaluate_baselines(small_model, X_small, y_small, 'SmallMLP-MNIST')
save_results(results_small, str(RESULTS_DIR / 'baselines_small_mnist.json'))

print('\nSmallMLP results:')
for method in ['fgsm', 'pgd']:
    s = results_small[method]['stats']
    print(f"  {method.upper():<6}  "
          f"ε_upper={s['mean_eps_upper']:.4f}±{s['std_eps_upper']:.4f}  "
          f"adv_found={s['fraction_adversarial_found']:.0%}  "
          f"time={s['mean_runtime_sec']:.2f}s/sample")

Loading small_mnist model...
  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}
Selected 100 correctly-classified samples



SmallMLP-MNIST: 100%|█████████████████████████████████████| 100/100 [00:30<00:00,  3.26sample/s, fgsm=0.281, pgd=0.188]

  Saved results → D:\concolic_exploration\results\baselines_small_mnist.json

SmallMLP results:
  FGSM    ε_upper=0.1640±0.1169  adv_found=100%  time=0.02s/sample
  PGD     ε_upper=0.1286±0.0547  adv_found=100%  time=0.28s/sample


## 6 — Run on MediumMLP (MNIST)
**Expected time: ~15 min**

In [7]:
print('Loading medium_mnist model...')
medium_model = load_model(str(MODELS_DIR / 'medium_mnist.pt'))

X_medium, y_medium = get_correctly_classified_samples(
    medium_model, X_mnist, y_mnist, N_SAMPLES, seed=SEED
)
print(f'Selected {len(X_medium)} correctly-classified samples\n')

results_medium = evaluate_baselines(medium_model, X_medium, y_medium, 'MediumMLP-MNIST')
save_results(results_medium, str(RESULTS_DIR / 'baselines_medium_mnist.json'))

print('\nMediumMLP results:')
for method in ['fgsm', 'pgd']:
    s = results_medium[method]['stats']
    print(f"  {method.upper():<6}  "
          f"ε_upper={s['mean_eps_upper']:.4f}±{s['std_eps_upper']:.4f}  "
          f"adv_found={s['fraction_adversarial_found']:.0%}  "
          f"time={s['mean_runtime_sec']:.2f}s/sample")

Loading medium_mnist model...
  Loaded ← D:\concolic_exploration\models\medium_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-256-256-256-10', 'best_test_acc': 0.9853, 'n_relu_neurons': 768, 'n_params': 335114}
Selected 100 correctly-classified samples



MediumMLP-MNIST: 100%|████████████████████████████████████| 100/100 [00:45<00:00,  2.17sample/s, fgsm=0.504, pgd=0.379]

  Saved results → D:\concolic_exploration\results\baselines_medium_mnist.json

MediumMLP results:
  FGSM    ε_upper=0.2631±0.1647  adv_found=99%  time=0.03s/sample
  PGD     ε_upper=0.1917±0.0812  adv_found=100%  time=0.43s/sample


## 7 — Run on LargeMLP (CIFAR-10)
**Expected time: ~20 min**

In [8]:
print('Loading large_cifar model...')
large_model = load_model(str(MODELS_DIR / 'large_cifar.pt'))

X_large, y_large = get_correctly_classified_samples(
    large_model, X_cifar, y_cifar, N_SAMPLES, seed=SEED
)
print(f'Selected {len(X_large)} correctly-classified samples\n')

results_large = evaluate_baselines(large_model, X_large, y_large, 'LargeMLP-CIFAR10')
save_results(results_large, str(RESULTS_DIR / 'baselines_large_cifar.json'))

print('\nLargeMLP results:')
for method in ['fgsm', 'pgd']:
    s = results_large[method]['stats']
    print(f"  {method.upper():<6}  "
          f"ε_upper={s['mean_eps_upper']:.4f}±{s['std_eps_upper']:.4f}  "
          f"adv_found={s['fraction_adversarial_found']:.0%}  "
          f"time={s['mean_runtime_sec']:.2f}s/sample")

Loading large_cifar model...
  Loaded ← D:\concolic_exploration\models\large_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'architecture': '3072-512-512-512-512-10', 'best_test_acc': 0.5593, 'n_relu_neurons': 2048, 'n_params': 2366474}
Selected 100 correctly-classified samples



LargeMLP-CIFAR10: 100%|███████████████████████████████████| 100/100 [05:05<00:00,  3.05s/sample, fgsm=0.011, pgd=0.010]

  Saved results → D:\concolic_exploration\results\baselines_large_cifar.json

LargeMLP results:
  FGSM    ε_upper=0.0665±0.0796  adv_found=100%  time=0.19s/sample
  PGD     ε_upper=0.0477±0.0430  adv_found=100%  time=2.86s/sample


## 8 — Summary across all models

In [9]:
summary = {
    'small_mnist'  : {m: results_small[m]['stats']  for m in ['fgsm', 'pgd']},
    'medium_mnist' : {m: results_medium[m]['stats'] for m in ['fgsm', 'pgd']},
    'large_cifar'  : {m: results_large[m]['stats']  for m in ['fgsm', 'pgd']},
}
save_results(summary, str(RESULTS_DIR / 'baselines_summary.json'))

# ── pretty table ──────────────────────────────────────────────────────────────
print()
print('═'*75)
print('  BASELINES SUMMARY')
print('═'*75)
print(f'  {"Model":<20} {"Method":<8} {"ε_upper (mean±std)":<22} '
      f'{"Adv%":>6} {"Time/sample":>12}')
print('─'*75)
for model_name, methods in summary.items():
    for method, s in methods.items():
        print(f"  {model_name:<20} {method.upper():<8} "
              f"{s['mean_eps_upper']:.4f} ± {s['std_eps_upper']:.4f}        "
              f"{s['fraction_adversarial_found']*100:>5.1f}% "
              f"{s['mean_runtime_sec']:>11.2f}s")
    print('─'*75)
print()
print('  Key observation: FGSM and PGD give ONLY upper bounds (ε_lower = 0).')
print('  Concolic exploration (Notebook 04) will add principled lower bounds.')
print()
print('  Next step → run 03_marabou_exact.ipynb')

  Saved results → D:\concolic_exploration\results\baselines_summary.json

═══════════════════════════════════════════════════════════════════════════
  BASELINES SUMMARY
═══════════════════════════════════════════════════════════════════════════
  Model                Method   ε_upper (mean±std)       Adv%  Time/sample
───────────────────────────────────────────────────────────────────────────
  small_mnist          FGSM     0.1640 ± 0.1169        100.0%        0.02s
  small_mnist          PGD      0.1286 ± 0.0547        100.0%        0.28s
───────────────────────────────────────────────────────────────────────────
  medium_mnist         FGSM     0.2631 ± 0.1647         99.0%        0.03s
  medium_mnist         PGD      0.1917 ± 0.0812        100.0%        0.43s
───────────────────────────────────────────────────────────────────────────
  large_cifar          FGSM     0.0665 ± 0.0796        100.0%        0.19s
  large_cifar          PGD      0.0477 ± 0.0430        100.0%        2.86s
─